In [3]:
!mkdir -p cuda bin data

In [4]:
!git clone https://github.com/Luis-AQR/CEPARCO_GRP3_IntegratingProject

Cloning into 'CEPARCO_GRP3_IntegratingProject'...
remote: Enumerating objects: 209, done.
remote: Counting objects: 100% (209/209), done.
remote: Compressing objects: 100% (147/147), done.
remote: Total 209 (delta 96), reused 106 (delta 41), pack-reused 0 (from 0)
Receiving objects: 100% (209/209), 31.23 MiB | 14.77 MiB/s, done.
Resolving deltas: 100% (96/96), done.


In [5]:
%%writefile sw_cuda.cu
/*
 * sw_cuda.cu - CUDA Accelerated Smith-Waterman (Week 10 Matrix Fill)
 * CEPARCO Integrating Project, Group 3
 *
 * This file implements the GPU version of the Smith-Waterman algorithm.
 * It uses a wavefront pattern, computing anti-diagonals in parallel.
 * The core logic and scoring are based on the sequential C implementation (sw_seq.c).
 *
 * Dependencies: CUDA Toolkit (nvcc).
 * Compile: nvcc -O3 sw_cuda.cu -o sw_cuda
 * Run: ./sw_cuda data/bench_1000.fasta --time --repeat 10
*/
#ifndef _CRT_SECURE_NO_WARNINGS
#define _CRT_SECURE_NO_WARNINGS
#endif

#if !defined(_WIN32) && !defined(_POSIX_C_SOURCE)
#define _POSIX_C_SOURCE 199309L
#endif

#include <ctype.h>
#include <errno.h>
#include <limits.h>
#include <stdio.h>
#include <stdlib.h>
#include <string.h>
#include <cuda_runtime.h>

#ifdef _WIN32
#ifndef WIN32_LEAN_AND_MEAN
#define WIN32_LEAN_AND_MEAN
#endif
#include <windows.h>
#else
#include <time.h>
#endif

// Default scoring logic
#define DEFAULT_MATCH 1
#define DEFAULT_MISMATCH 1
#define DEFAULT_GAP 2
#define PRINT_LIMIT 40
#define MAX_SEQ_LEN 46000
#define ALIGN_LINE 60
#define MAX_REPEAT 1000
#define IDX(i, j, ncols) ((size_t)(i) * (size_t)(ncols) + (size_t)(j))

// Taken from sw_seq.c to hold scoring parameters
typedef struct {
    int match;
    int mismatch;
    int gap;
} Scoring;

// Taken from sw_seq.c to hold the results of the matrix fill
typedef struct {
    int score;
    int i, j;
    unsigned long long checksum;
} FillResult;

// Taken from sw_seq.c to hold the reconstructed traceback alignment
typedef struct {
    char *buf_a;
    char *buf_b;
    char *a_aln;
    char *b_aln;
    int len;
    int i_start, i_end;
    int j_start, j_end;
    int matches, mismatches, gaps;
} Alignment;

// Tracking execution times
typedef struct {
    double alloc, fill, trace, total;
} RunTime;

// From sw_seq.c: High-res timer
static double now_seconds(void) {
#ifdef _WIN32
    LARGE_INTEGER freq, now;
    QueryPerformanceFrequency(&freq);
    QueryPerformanceCounter(&now);
    return (double)now.QuadPart / (double)freq.QuadPart;
#else
    struct timespec ts;
    clock_gettime(CLOCK_MONOTONIC, &ts);
    return (double)ts.tv_sec + (double)ts.tv_nsec * 1e-9;
#endif
}

// From sw_seq.c: Reads exactly two sequences from a FASTA file into allocated buffers
static int read_fasta_pair(const char *path, char **out_a, char **out_b) {
    FILE *f = fopen(path, "r");
    if (!f) {
        fprintf(stderr, "error: cannot open '%s'\n", path);
        return -1;
    }
    char *seq[2] = {NULL, NULL};
    size_t len[2] = {0, 0}, cap[2] = {0, 0};
    int nrec = 0;
    int cur = -1;
    size_t buf_size = 4096;
    char *line = (char *)malloc(buf_size);
    if (!line) { fclose(f); return -1; }

    for (;;) {
        size_t used = 0;
        int got_any = 0;
        for (;;) {
            if (used + 2 > buf_size) {
                buf_size *= 2;
                char *tmp = (char *)realloc(line, buf_size);
                if (!tmp) { free(line); free(seq[0]); free(seq[1]); fclose(f); return -1; }
                line = tmp;
            }
            if (!fgets(line + used, (int)(buf_size - used), f)) break;
            got_any = 1;
            used += strlen(line + used);
            if (used > 0 && line[used - 1] == '\n') break;
        }
        if (!got_any) break;
        while (used > 0 && (line[used - 1] == '\n' || line[used - 1] == '\r' ||
                            line[used - 1] == ' ' || line[used - 1] == '\t'))
            line[--used] = '\0';
        if (used == 0) continue;

        if (line[0] == '>') {
            if (nrec == 2) goto fail;
            cur = nrec++;
            continue;
        }
        if (cur < 0) goto fail;
        for (size_t k = 0; k < used; k++) {
            unsigned char ch = (unsigned char)line[k];
            if (!isalpha(ch)) goto fail;
            if (len[cur] + 2 > cap[cur]) {
                cap[cur] = cap[cur] ? cap[cur] * 2 : 1024;
                char *tmp = (char *)realloc(seq[cur], cap[cur]);
                if (!tmp) goto fail;
                seq[cur] = tmp;
            }
            seq[cur][len[cur]++] = (char)toupper(ch);
            seq[cur][len[cur]] = '\0';
        }
    }
    if (nrec != 2 || len[0] == 0 || len[1] == 0) goto fail;
    free(line); fclose(f);
    *out_a = seq[0]; *out_b = seq[1];
    return 0;
fail:
    free(line); free(seq[0]); free(seq[1]); fclose(f);
    return -1;
}

// --- CUDA Kernels ---
// Computes a single anti-diagonal (wavefront) of the scoring matrix in parallel.
__global__ void fill_diag(const char *a, const char *b, int N, int d, int i_min, int count,
                          int *H, int match, int mismatch, int gap)
{
    int t = blockIdx.x * blockDim.x + threadIdx.x; // Calculate global thread ID
    if (t >= count) return; // Prevent out-of-bounds execution for this diagonal

    // Map thread ID to cell coordinates (i, j) on the current anti-diagonal 'd'
    int i = i_min + t, j = d - i;
    size_t nc = (size_t)N + 1;

    // Calculate the dynamic programming candidates (match/mismatch, gap up, gap left)
    int s    = (a[i - 1] == b[j - 1]) ? match : -mismatch;
    int diag = H[(size_t)(i - 1) * nc + (j - 1)] + s;
    int up   = H[(size_t)(i - 1) * nc + j] - gap;
    int left = H[(size_t)i * nc + (j - 1)] - gap;

    // Store the max of (0, diag, up, left) in the global memory matrix
    H[(size_t)i * nc + j] = max(max(0, diag), max(up, left));
}

// Performs a parallel reduction using shared memory to find the highest score and total checksum.
__global__ void reduce_best(const int *H, size_t cells, unsigned long long *best_key,
                            unsigned long long *sum)
{
    __shared__ unsigned long long s_key[256], s_sum[256]; // Shared memory for block-level reduction
    unsigned long long key = 0, acc = 0;

    // Grid-stride loop
    for (size_t k = (size_t)blockIdx.x * blockDim.x + threadIdx.x; k < cells;
         k += (size_t)gridDim.x * blockDim.x) {
        int h = H[k];
        acc += (unsigned)h;
        if (h > 0) {
            // Pack the score and cell index into a single 64-bit integer to find the *first* max safely
            unsigned long long kk = ((unsigned long long)h << 32) | (unsigned)(~(unsigned)k);
            if (kk > key) key = kk;
        }
    }
    s_key[threadIdx.x] = key;
    s_sum[threadIdx.x] = acc;
    __syncthreads(); // Wait for all threads in the block to finish loading shared memory

    // Tree-based reduction within the block
    for (int s = blockDim.x / 2; s > 0; s >>= 1) {
        if ((int)threadIdx.x < s) {
            if (s_key[threadIdx.x + s] > s_key[threadIdx.x]) s_key[threadIdx.x] = s_key[threadIdx.x + s];
            s_sum[threadIdx.x] += s_sum[threadIdx.x + s];
        }
        __syncthreads();
    }

    // If thread 0 writes the blocks final reduced values to global memory
    if (threadIdx.x == 0) { atomicMax(best_key, s_key[0]); atomicAdd(sum, s_sum[0]); }
}

// --- GPU Execution Wrapper ---
// Manages device memory, executes kernel launches, and retrieves final results to the host.
static FillResult fill_matrix_cuda(const char *a, int M, const char *b, int N, int *h_H, Scoring sc, double *fill_time) {
    char *d_a, *d_b;
    int *d_H;
    unsigned long long *d_best_key, *d_sum;
    size_t num_cells = (size_t)(M + 1) * (N + 1);

    // Allocate global device memory for the sequences and the full scoring matrix
    cudaMalloc((void**)&d_a, M);
    cudaMalloc((void**)&d_b, N);
    cudaMalloc((void**)&d_H, num_cells * sizeof(int));
    cudaMalloc((void**)&d_best_key, sizeof(unsigned long long));
    cudaMalloc((void**)&d_sum, sizeof(unsigned long long));

    // Zero-initialize device memory for the scoring matrix
    cudaMemset(d_H, 0, num_cells * sizeof(int));
    cudaMemset(d_best_key, 0, sizeof(unsigned long long));
    cudaMemset(d_sum, 0, sizeof(unsigned long long));

    // Copy sequences from Host to Device
    cudaMemcpy(d_a, a, M, cudaMemcpyHostToDevice);
    cudaMemcpy(d_b, b, N, cudaMemcpyHostToDevice);
    cudaDeviceSynchronize();

    double t1 = now_seconds(); // Start timing the core calculation

    // Anti-diagonal fill loop
    int max_d = M + N;
    for (int d = 2; d <= max_d; d++) {
        int i_min = max(1, d - N);
        int i_max = min(M, d - 1);
        int count = i_max - i_min + 1; // Number of cells on this diagonal

        if (count > 0) {
            int threads = 256;
            int blocks = (count + threads - 1) / threads;
            // Execute the anti-diagonal on the GPU
            fill_diag<<<blocks, threads>>>(d_a, d_b, N, d, i_min, count, d_H, sc.match, sc.mismatch, sc.gap);
        }
    }

    // Checksum and best cell reduction
    int red_threads = 256;
    int red_blocks = min(1024, (int)((num_cells + red_threads - 1) / red_threads));
    reduce_best<<<red_blocks, red_threads>>>(d_H, num_cells, d_best_key, d_sum);
    cudaDeviceSynchronize(); // Wait for all kernels to finish

    *fill_time = now_seconds() - t1; // End timing

    // Retrieves results
    unsigned long long h_best_key, h_sum;
    cudaMemcpy(&h_best_key, d_best_key, sizeof(unsigned long long), cudaMemcpyDeviceToHost);
    cudaMemcpy(&h_sum, d_sum, sizeof(unsigned long long), cudaMemcpyDeviceToHost);
    cudaMemcpy(h_H, d_H, num_cells * sizeof(int), cudaMemcpyDeviceToHost);

    // Free memory
    cudaFree(d_a);
    cudaFree(d_b);
    cudaFree(d_H);
    cudaFree(d_best_key);
    cudaFree(d_sum);

    // Unpack the best score and coordinates from the 64-bit reduction key
    FillResult r;
    r.score = (int)(h_best_key >> 32);
    if (r.score > 0) {
        size_t k = (size_t)(~((unsigned int)(h_best_key & 0xFFFFFFFF)));
        r.i = (int)(k / (N + 1));
        r.j = (int)(k % (N + 1));
    } else {
        r.i = 0;
        r.j = 0;
    }
    r.checksum = h_sum;
    return r;
}

// --- Independent Host Traceback and Printing ---
// From sw_seq.c: Frees the traceback buffers
static void free_alignment(Alignment *al) {
    free(al->buf_a); free(al->buf_b);
    al->buf_a = al->buf_b = al->a_aln = al->b_aln = NULL;
}

// From sw_seq.c: Traces back from the best cell to reconstruct the optimal alignment path on the CPU
static int traceback(const char *a, const char *b, int M, int N, const int *H,
                     Scoring sc, int bi, int bj, Alignment *out) {
    const int ncols = N + 1;
    const size_t cap = (size_t)M + (size_t)N + 1;
    memset(out, 0, sizeof *out);
    out->buf_a = (char *)malloc(cap + 1);
    out->buf_b = (char *)malloc(cap + 1);
    if (!out->buf_a || !out->buf_b) { free_alignment(out); return -1; }
    size_t pos = cap;
    out->buf_a[pos] = '\0'; out->buf_b[pos] = '\0';
    int i = bi, j = bj;
    out->i_end = bi; out->j_end = bj;

    while (H[IDX(i, j, ncols)] > 0) {
        const int h = H[IDX(i, j, ncols)];
        if (i > 0 && j > 0) {
            int s = (a[i - 1] == b[j - 1]) ? sc.match : -sc.mismatch;
            if (h == H[IDX(i - 1, j - 1, ncols)] + s) {
                --pos; out->buf_a[pos] = a[i - 1]; out->buf_b[pos] = b[j - 1];
                if (a[i - 1] == b[j - 1]) out->matches++; else out->mismatches++;
                i--; j--; continue;
            }
        }
        if (i > 0 && h == H[IDX(i - 1, j, ncols)] - sc.gap) {
            --pos; out->buf_a[pos] = a[i - 1]; out->buf_b[pos] = '-'; out->gaps++; i--; continue;
        }
        if (j > 0 && h == H[IDX(i, j - 1, ncols)] - sc.gap) {
            --pos; out->buf_a[pos] = '-'; out->buf_b[pos] = b[j - 1]; out->gaps++; j--; continue;
        }
        free_alignment(out); return -1;
    }
    out->i_start = i + 1; out->j_start = j + 1;
    out->len = (int)(cap - pos);
    out->a_aln = out->buf_a + pos; out->b_aln = out->buf_b + pos;
    return 0;
}

// From sw_seq.c: Independently verifies the traceback sequence strings match the computed max score
static int rescore_alignment(const Alignment *al, const char *a, const char *b, Scoring sc) {
    if (!al->a_aln || !al->b_aln) return INT_MIN;
    if ((int)strlen(al->a_aln) != al->len || (int)strlen(al->b_aln) != al->len) return INT_MIN;
    int score = 0, ia = al->i_start - 1, jb = al->j_start - 1;
    for (int k = 0; k < al->len; k++) {
        char x = al->a_aln[k], y = al->b_aln[k];
        if (x == '-' && y == '-') return INT_MIN;
        if (x == '-') { if (b[jb] != y) return INT_MIN; jb++; score -= sc.gap; }
        else if (y == '-') { if (a[ia] != x) return INT_MIN; ia++; score -= sc.gap; }
        else { if (a[ia] != x || b[jb] != y) return INT_MIN; ia++; jb++; score += (x == y) ? sc.match : -sc.mismatch; }
    }
    if (ia != al->i_end || jb != al->j_end) return INT_MIN;
    return score;
}

// From sw_seq.c: Prints the aligned sequences to stdout
static void print_alignment(const Alignment *al) {
    printf("Alignment: length %d (%d matches, %d mismatches, %d gaps)\n", al->len, al->matches, al->mismatches, al->gaps);
    printf("Range    : a[%d..%d], b[%d..%d]\n", al->i_start, al->i_end, al->j_start, al->j_end);
    int ia = al->i_start, jb = al->j_start;
    for (int k = 0; k < al->len; k += ALIGN_LINE) {
        int n = al->len - k < ALIGN_LINE ? al->len - k : ALIGN_LINE;
        printf("\n  a %6d  %.*s\n            ", ia, n, al->a_aln + k);
        for (int c = 0; c < n; c++) { char x = al->a_aln[k + c], y = al->b_aln[k + c]; putchar((x == y && x != '-') ? '|' : ' '); }
        printf("\n  b %6d  %.*s\n", jb, n, al->b_aln + k);
        for (int c = 0; c < n; c++) { if (al->a_aln[k + c] != '-') ia++; if (al->b_aln[k + c] != '-') jb++; }
    }
    printf("\n");
}

// From sw_seq.c: Prints the full dynamic programming matrix
static void print_matrix(const char *b, int N, const char *a, int M, const int *H) {
    const int ncols = N + 1;
    printf("\nScore matrix H (%d x %d):\n\n      ", M + 1, N + 1);
    for (int j = 0; j <= N; j++) printf("%4d", j);
    printf("\n      %4s", "");
    for (int j = 1; j <= N; j++) printf("%4c", b[j - 1]);
    printf("\n");
    for (int i = 0; i <= M; i++) {
        printf("%3d %c ", i, i == 0 ? ' ' : a[i - 1]);
        for (int j = 0; j <= N; j++) printf("%4d", H[IDX(i, j, ncols)]);
        printf("\n");
    }
}

// Parses string arguments to ensure they are valid positive integers
static int parse_positive(const char *text, int *out) {
    char *end = NULL; errno = 0;
    long v = strtol(text, &end, 10);
    if (errno != 0 || end == text || *end != '\0' || v < 1 || v > 1000000) return -1;
    *out = (int)v; return 0;
}

// From sw_seq.c: Safely zero memory without compiler optimizations dropping the call
static void *(*volatile zero_memory_fn)(void *, int, size_t) = memset;
static void zero_memory(void *p, size_t n) { zero_memory_fn(p, 0, n); }

// Helper for qsort to sort doubles
static int cmp_double(const void *x, const void *y) {
    double a = *(const double *)x, b = *(const double *)y; return (a > b) - (a < b);
}

// Calculates statistics from benchmark timing array
static void stats(double *v, int n, double *mn, double *med, double *mean) {
    qsort(v, (size_t)n, sizeof(double), cmp_double);
    double sum = 0.0; for (int k = 0; k < n; k++) sum += v[k];
    *mn = v[0]; *med = (n % 2) ? v[n / 2] : 0.5 * (v[n / 2 - 1] + v[n / 2]); *mean = sum / n;
}

// Main
int main(int argc, char **argv) {
    if (argc < 2 || strcmp(argv[1], "--help") == 0 || strcmp(argv[1], "-h") == 0) return 2;

    Scoring sc = {DEFAULT_MATCH, DEFAULT_MISMATCH, DEFAULT_GAP};
    int show_matrix = 1, do_traceback = 1, repeat = 1, warmup = 0, want_csv = 0, want_time = 0;
    for (int k = 2; k < argc; k++) {
        int *target = NULL;
        if (strcmp(argv[k], "--no-matrix") == 0) { show_matrix = 0; continue; }
        else if (strcmp(argv[k], "--no-traceback") == 0) { do_traceback = 0; continue; }
        else if (strcmp(argv[k], "--time") == 0) { want_time = 1; continue; }
        else if (strcmp(argv[k], "--warmup") == 0) { warmup = 1; continue; }
        else if (strcmp(argv[k], "--csv") == 0) { want_csv = 1; continue; }
        else if (strcmp(argv[k], "--repeat") == 0) target = &repeat;
        else if (strcmp(argv[k], "--match") == 0) target = &sc.match;
        else if (strcmp(argv[k], "--mismatch") == 0) target = &sc.mismatch;
        else if (strcmp(argv[k], "--gap") == 0) target = &sc.gap;
        if (k + 1 >= argc || parse_positive(argv[k + 1], target) != 0) return 2;
        k++;
    }
    const int show_timing = want_time || want_csv || warmup || repeat > 1;

    char *a = NULL, *b = NULL;
    if (read_fasta_pair(argv[1], &a, &b) != 0) return 1;

    size_t Ms = strlen(a), Ns = strlen(b);
    int M = (int)Ms, N = (int)Ns;
    size_t cells = (size_t)(M + 1) * (size_t)(N + 1);
    double mb = (double)cells * sizeof(int) / (1024.0 * 1024.0);

    int status = 0, printed = 0;
    RunTime times[MAX_REPEAT + 1];

    for (int run = warmup ? 0 : 1; run <= repeat; run++) {
        double t0 = now_seconds();

        // Host allocation
        int *h_H = (int *)malloc(cells * sizeof(int));
        if (h_H) zero_memory(h_H, cells * sizeof(int));
        double t1 = now_seconds();

        double fill_duration = 0.0;

        // Execute the CUDA kernel pipeline
        FillResult res = fill_matrix_cuda(a, M, b, N, h_H, sc, &fill_duration);

        double t2 = now_seconds();

        Alignment al; int have_al = 0, tb_failed = 0;

        // Perform sequential CPU traceback
        if (res.score > 0 && do_traceback) {
            if (traceback(a, b, M, N, h_H, sc, res.i, res.j, &al) == 0) have_al = 1;
            else tb_failed = 1;
        }
        double t3 = now_seconds();

        // Record metrics
        times[run].alloc = (t1 - t0);
        times[run].fill = fill_duration;
        times[run].trace = t3 - t2;
        times[run].total = (t3 - t0) - (t2 - t1) + fill_duration;

        if (tb_failed) status = 1;

        if (!printed) {
            printed = 1;
            printf("Smith-Waterman (CUDA Colab Version)\n");
            printf("File     : %s\nLengths  : M = %d (rows), N = %d (columns)\n", argv[1], M, N);
            printf("Scoring  : match +%d, mismatch -%d, gap -%d\n", sc.match, sc.mismatch, sc.gap);
            printf("Matrix   : %llu cells, %.1f MB\n", (unsigned long long)cells, mb);

            if (show_matrix && M <= PRINT_LIMIT && N <= PRINT_LIMIT) print_matrix(b, N, a, M, h_H);

            printf("\nMax score: %d\n", res.score);
            if (res.score > 0) printf("Max cell : H(%d, %d)\n", res.i, res.j);
            else printf("Max cell : none (no local alignment, every cell is 0)\n");

            if (have_al) {
                printf("\n"); print_alignment(&al);
                int rs = rescore_alignment(&al, a, b, sc);
                if (rs == res.score) printf("Re-score : %d (OK)\n", rs);
                else { printf("Re-score : %d (MISMATCH, expected %d)\n", rs, res.score); status = 1; }
            }
            printf("Checksum : %llu (sum of all matrix cells)\n", res.checksum);
        }
        if (have_al) free_alignment(&al);
        free(h_H);
    }

    if (show_timing && printed) {
        double v[MAX_REPEAT], mn, med, mean;
        printf("\nTiming   : %d run(s)\n  %-4s %10s %10s %10s %10s   (seconds)\n", repeat, "run", "alloc", "fill", "traceback", "total");
        for (int run = warmup ? 0 : 1; run <= repeat; run++)
            printf("  %-4d %10.6f %10.6f %10.6f %10.6f\n", run, times[run].alloc, times[run].fill, times[run].trace, times[run].total);
        if (repeat >= 1) {
            for (int k = 0; k < repeat; k++) v[k] = times[1 + k].fill;
            stats(v, repeat, &mn, &med, &mean);
            printf("  fill      min %.6f  median %.6f  mean %.6f\n", mn, med, mean);
        }
    }
    free(a);
    free(b);
    return status;
}

Overwriting sw_cuda.cu


In [6]:
!nvcc -O3 sw_cuda.cu -o sw_cuda

In [7]:
!./sw_cuda CEPARCO_GRP3_IntegratingProject/smith-waterman/data/bench_1000.fasta --time --repeat 10

Smith-Waterman (CUDA Colab Version)
File     : CEPARCO_GRP3_IntegratingProject/smith-waterman/data/bench_1000.fasta
Lengths  : M = 1000 (rows), N = 1000 (columns)
Scoring  : match +1, mismatch -1, gap -2
Matrix   : 1002001 cells, 3.8 MB

Max score: 160
Max cell : H(784, 995)

Alignment: length 225 (198 matches, 16 mismatches, 11 gaps)
Range    : a[566..784], b[776..995]

  a    566  GTGGTCTGTCGCCTCAGCCCAGATTTTCA-TATTATGCAGAAAATCTACTTCG-CCTGAT
            || | || ||||  |   | ||||||||| ||||||||||||||||||||||| ||||||
  b    776  GTTGACTTTCGCGCCCCTCTAGATTTTCACTATTATGCAGAAAATCTACTTCGCCCTGAT

  a    624  ACGAGTCGGTTATCTTCGGATACTGTATAGTCCCACCTGGTGATCCTATGC-T-TGTGAG
            ||||| || || |||||||||||||||||  |||| ||||||||||||||| | ||||||
  b    836  ACGAG-CGCTT-TCTTCGGATACTGTATA-CCCCA-CTGGTGATCCTATGCTTGTGTGAG

  a    682  TACCCAGAAAATAGCGACGGA-CCGCGGTGTTAAGTGTCGAGCTACATCACTTCTCATGT
            |||||||| |||| ||||||| ||||||||||||||||||||||| ||||||||| ||||
  b    892  TACCCAGACAATAACGACGGACCCGCGG

In [8]:
!./sw_cuda CEPARCO_GRP3_IntegratingProject/smith-waterman/data/bench_5000.fasta --time --repeat 10

Smith-Waterman (CUDA Colab Version)
File     : CEPARCO_GRP3_IntegratingProject/smith-waterman/data/bench_5000.fasta
Lengths  : M = 5000 (rows), N = 5000 (columns)
Scoring  : match +1, mismatch -1, gap -2
Matrix   : 25010001 cells, 95.4 MB

Max score: 805
Max cell : H(2313, 1448)

Alignment: length 1043 (954 matches, 29 mismatches, 60 gaps)
Range    : a[1298..2313], b[439..1448]

  a   1298  GTATC-AG--GCTATGTCAC-CAGATTTTCATATTATGCAGAAAATCTACTT-CGCC-TG
            ||||| ||  | || | ||| |||| |||||||||||||||||||||||||| |||| ||
  b    439  GTATCAAGTAGGTA-GCCACTCAGA-TTTCATATTATGCAGAAAATCTACTTACGCCTTG

  a   1352  ATA-CGAGTCGGTTATCTTCGGATACTGTATAGTCCCACCTGGTGAT-CCTATGCTTGTG
            ||| ||||||||||||||||||||| |||||||||||||||||| || ||||||||||||
  b    497  ATAGCGAGTCGGTTATCTTCGGATA-TGTATAGTCCCACCTGGTCATCCCTATGCTTGTG

  a   1410  AGTACCCAGAAAATAGCGACGGACCGCGGTGTTAAGTGTCGAGCTACATCACTTCTCATG
            ||||||| |||| ||||| | | ||||||||||||||||| |||||||||||||||||||
  b    556  AGTACCCTGAAAGTAGCG-C

In [9]:
!./sw_cuda CEPARCO_GRP3_IntegratingProject/smith-waterman/data/bench_10000.fasta --time --repeat 10

Smith-Waterman (CUDA Colab Version)
File     : CEPARCO_GRP3_IntegratingProject/smith-waterman/data/bench_10000.fasta
Lengths  : M = 10000 (rows), N = 10000 (columns)
Scoring  : match +1, mismatch -1, gap -2
Matrix   : 100020001 cells, 381.5 MB

Max score: 1536
Max cell : H(5403, 9807)

Alignment: length 2069 (1863 matches, 85 mismatches, 121 gaps)
Range    : a[3387..5403], b[7808..9807]

  a   3387  CAGATTTTCATATTATGCAGAAAATCTACTTCGCCTGATACGAGTCGGTTATCTTCGGAT
            |||| ||||||| ||||||||||||||||||||||||||||| || | ||||||||||||
  b   7808  CAGA-TTTCATACTATGCAGAAAATCTACTTCGCCTGATACG-GT-GCTTATCTTCGGAT

  a   3447  ACTGTATAGTCCCA-CCTGGTGATCCTATGCTTGTGAGTACCCAGAAAATAGCGACGGAC
            ||||||| |||||| ||||||||||||||||||||||||||| || ||||||||||| ||
  b   7865  ACTGTAT-GTCCCATCCTGGTGATCCTATGCTTGTGAGTACCAAG-AAATAGCGACGCAC

  a   3506  CGCGGT-GTTAAGTGTCGAGCTACATCACTT-CTCATGTAGCCAGAAGGCTGCAACTCAT
            || ||| ||||||||||||||||||| | || |||| ||||||||||||| |||||||| 
  b   7923  CGTGGTAGTTA